<a href="https://colab.research.google.com/github/Ranjitkumarmuduli/crop-disease-detection/blob/main/02_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install timm kagglehub
import os, time, copy, torch, timm, pandas as pd, numpy as np, torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from google.colab import drive
import kagglehub
drive.mount('/content/drive')
kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")

ROOT  = "/kaggle/input/plantvillage-dataset/plantvillage dataset/color"
PROJ  = "/content/drive/MyDrive/crop_disease_project"
SPLIT = f"{PROJ}/splits"
os.makedirs(f"{PROJ}/weights", exist_ok=True); os.makedirs(f"{PROJ}/results", exist_ok=True)
DEVICE = "cuda"
print(torch.cuda.get_device_name(0))

Mounted at /content/drive
Using Colab cache for faster access to the 'plantvillage-dataset' dataset.
Tesla T4


In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = T.Compose([T.RandomResizedCrop(224, scale=(0.7, 1.0)), T.RandomHorizontalFlip(),
                      T.RandomVerticalFlip(), T.RandomRotation(20), T.ColorJitter(0.2, 0.2, 0.2),
                      T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf  = T.Compose([T.Resize((224, 224)), T.ToTensor(), T.Normalize(MEAN, STD)])

class Leaf(Dataset):
    def __init__(self, csv, tf): self.d, self.tf = pd.read_csv(csv), tf
    def __len__(self): return len(self.d)
    def __getitem__(self, i):
        r = self.d.iloc[i]
        return self.tf(Image.open(f"{ROOT}/{r.relpath}").convert("RGB")), int(r.label)

CLASSES = pd.read_csv(f"{SPLIT}/classes.csv", header=None)[0].tolist()
kw = dict(batch_size=64, num_workers=2, pin_memory=True)
train_dl = DataLoader(Leaf(f"{SPLIT}/train.csv", train_tf), shuffle=True, drop_last=True, **kw)
val_dl   = DataLoader(Leaf(f"{SPLIT}/val.csv", eval_tf), **kw)
test_dl  = DataLoader(Leaf(f"{SPLIT}/test.csv", eval_tf), **kw)
print(len(CLASSES), "classes |", len(train_dl), "train batches")

38 classes | 593 train batches


In [ ]:
MODEL = "efficientnet_b0"     # later: "resnet50", "mobilenetv3_large_100"
EPOCHS_HEAD, EPOCHS_FULL = 1, 5

def run_epoch(model, dl, crit, opt=None, scaler=None):
    model.train(opt is not None); n = correct = loss_sum = 0
    with torch.set_grad_enabled(opt is not None):
        for x, y in dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.autocast("cuda", dtype=torch.float16):
                out = model(x); loss = crit(out, y)
            if opt:
                opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            loss_sum += loss.item() * len(y); correct += (out.argmax(1) == y).sum().item(); n += len(y)
    return loss_sum / n, correct / n

torch.manual_seed(42)
model = timm.create_model(MODEL, pretrained=True, num_classes=len(CLASSES)).to(DEVICE)
crit, scaler = nn.CrossEntropyLoss(label_smoothing=0.1), torch.amp.GradScaler("cuda")
head = set(id(p) for p in model.get_classifier().parameters())
hist, best = [], 1e9
for stage, epochs, lr in [("head", EPOCHS_HEAD, 1e-3), ("full", EPOCHS_FULL, 3e-4)]:
    for p in model.parameters(): p.requires_grad = stage == "full" or id(p) in head
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    for ep in range(epochs):
        t = time.time()
        tl, ta = run_epoch(model, train_dl, crit, opt, scaler); vl, va = run_epoch(model, val_dl, crit); sched.step()
        hist.append(dict(stage=stage, train_loss=tl, train_acc=ta, val_loss=vl, val_acc=va))
        print(f"{stage} {ep+1}/{epochs}  train {ta:.4f}  val {va:.4f}  loss {vl:.3f}  ({(time.time()-t)/60:.1f} min)")
        if vl < best: best = vl; torch.save(model.state_dict(), f"{PROJ}/weights/{MODEL}.pt")
pd.DataFrame(hist).to_csv(f"{PROJ}/results/history_{MODEL}.csv", index=False)

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

head 1/1  train 0.7590  val 0.9034  loss 1.097  (6.5 min)
full 1/5  train 0.9734  val 0.9881  loss 0.751  (4.7 min)
full 2/5  train 0.9935  val 0.9945  loss 0.712  (4.6 min)
full 3/5  train 0.9957  val 0.9968  loss 0.697  (4.6 min)
full 4/5  train 0.9980  val 0.9983  loss 0.690  (4.9 min)
full 5/5  train 0.9991  val 0.9984  loss 0.687  (4.8 min)


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
model.load_state_dict(torch.load(f"{PROJ}/weights/{MODEL}.pt")); model.eval()
ys, ps = [], []
with torch.no_grad():
    for x, y in test_dl:
        with torch.autocast("cuda", dtype=torch.float16): ps += model(x.to(DEVICE)).argmax(1).cpu().tolist()
        ys += y.tolist()
print(f"{MODEL}: accuracy {accuracy_score(ys, ps):.4f} | macro F1 {f1_score(ys, ps, average='macro'):.4f} | "
      f"params {sum(p.numel() for p in model.parameters())/1e6:.1f}M")
pd.DataFrame(classification_report(ys, ps, target_names=CLASSES, output_dict=True)).T.to_csv(f"{PROJ}/results/report_{MODEL}.csv")

efficientnet_b0: accuracy 0.9978 | macro F1 0.9964 | params 4.1M
